# FHIR Silver SCD Type 2


In [ ]:
api_base = 'https://hapi.fhir.org/baseR4'


## Spark SQL history and PySpark resource columns


In [ ]:
from pyspark.sql import functions as F
RESOURCES = ('Patient', 'Encounter', 'Observation', 'Condition')

def table_name(layer, resource):
    return f"{layer}.fhir_{resource.lower()}"


def silver_history(spark, bronze):
    """Build business-change intervals in source order using Spark SQL."""
    return spark.sql("""
        WITH ranked AS (
            SELECT *, ROW_NUMBER() OVER (
                PARTITION BY resource_id, source_last_updated
                ORDER BY TRY_CAST(source_version_id AS DECIMAL(38, 0)) DESC NULLS LAST,
                         source_version_id DESC NULLS LAST, event_id DESC
            ) AS choice
            FROM {bronze}
        ),
        compared AS (
            SELECT *, LAG(content_hash) OVER (
                PARTITION BY resource_id ORDER BY source_last_updated
            ) AS previous_hash
            FROM ranked WHERE choice = 1
        ),
        changes AS (
            SELECT *, LEAD(source_last_updated) OVER (
                PARTITION BY resource_id ORDER BY source_last_updated
            ) AS valid_to
            FROM compared WHERE NOT (content_hash <=> previous_hash)
        )
        SELECT resource_type, source_date, run_id, attempt_id, relative_path,
               extraction_timestamp, raw_saved_timestamp, api_url_or_params,
               resource_id, source_version_id, source_last_updated, resource_json,
               content_hash, event_id, source_last_updated AS valid_from, valid_to,
               valid_to IS NULL AS is_current, CURRENT_TIMESTAMP() AS silver_saved_timestamp
        FROM changes
    """, bronze=bronze)


def json_field(path):
    return F.get_json_object("resource_json", path)


def fhir_instant(path):
    value = json_field(path)
    return F.when(value.rlike(r"T.*(Z|[+-][0-9]{2}:[0-9]{2})$"),
                  F.try_to_timestamp(value))


def reference_id(path, target_type, api_base):
    reference = json_field(path)
    relative = F.when(reference.startswith(api_base.rstrip("/") + "/"),
                      F.substring(reference, len(api_base.rstrip("/")) + 2, 2000)).otherwise(reference)
    match = F.regexp_extract(relative, rf"^{target_type}/([A-Za-z0-9.-]{{1,64}})(?:/_history/[^/]+)?$", 1)
    return F.when(match != "", match)


def model_silver(history, resource, api_base):
    if resource == "Patient":
        return history.withColumn("gender", F.lower(F.trim(json_field("$.gender"))))
    out = (history.withColumn("patient_id", reference_id("$.subject.reference", "Patient", api_base))
           .withColumn("status", F.lower(F.trim(json_field("$.status")))))
    if resource == "Encounter":
        return (out.withColumn("encounter_start_text", json_field("$.period.start"))
                .withColumn("encounter_start_utc", fhir_instant("$.period.start")))
    out = (out.withColumn("encounter_id", reference_id("$.encounter.reference", "Encounter", api_base))
           .withColumn("code", json_field("$.code.coding[0].code"))
           .withColumn("code_system", json_field("$.code.coding[0].system")))
    if resource == "Observation":
        return (out.withColumn("effective_datetime_text", json_field("$.effectiveDateTime"))
               .withColumn("effective_at_utc", fhir_instant("$.effectiveDateTime"))
               .withColumn("value_quantity_text", json_field("$.valueQuantity.value"))
               .withColumn("value_unit", json_field("$.valueQuantity.unit")))
    return out.withColumn("status", F.lower(F.trim(json_field("$.clinicalStatus.coding[0].code"))))


def build_silver(spark, api_base):
    spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
    for resource in RESOURCES:
        bronze = spark.table(table_name("bronze", resource))
        silver = model_silver(silver_history(spark, bronze), resource, api_base)
        silver.write.format("delta").mode("overwrite").saveAsTable(table_name("silver", resource))




In [ ]:
spark.conf.set('spark.sql.session.timeZone', 'UTC')
build_silver(spark, api_base)
